# Часть 3 из 8. Первое сравнение моделей: CatBoost, LightGBM, XGBoost

**Проект:** прогноз цены квартиры на вторичном рынке Москвы по датасету `Moscow_Real_Estate`.

**В этой части:**
- обучение CatBoost, LightGBM и XGBoost на логарифме цены;
- метрики MAE, RMSE, MAPE и R² на валидации, сравнение с нейросетью FlatNet;
- взвешенное среднее трёх моделей.

**Главный результат:** CatBoost — MAPE 6,68 %, LightGBM — 7,15 %, XGBoost — 7,97 %, FlatNet — 8,17 %; среднее трёх бустингов — 6,79 %.

> Код продолжает предыдущие части и использует их переменные. Результаты уже сохранены в блокноте, поэтому его можно читать без запуска. Чтобы пересчитать часть, сначала выполните в этом же ядре части 1–2 по порядку: `%run -i 01_eda.ipynb` и так далее.

← [Часть 2. Подготовка признаков и нормализация](02_features.ipynb) · [Все части](README.md) · [Часть 4. Эксперименты с CatBoost](04_catboost_experiments.ipynb) →

## Модель если чо пока что все модели чисто дипсик в соло сделал, потом буду смотреть и править

In [41]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", device)

Device: cuda


In [42]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")

PyTorch: 2.6.0+cu124
CUDA available: True
GPU: NVIDIA GeForce RTX 4060


In [43]:
corr = df_sec_m.corr(method='spearman', numeric_only=True)['price_rub']
print(corr.sort_values(key=abs, ascending=False).head(30))

price_rub                   1.000000
to_center_km               -0.678292
metro_distance_min         -0.652246
total_area                  0.420567
living_area                 0.409018
metro_distance_type         0.387308
rooms                       0.367675
lat                         0.263167
lon                         0.224219
kitchen_area                0.192598
months_since_start          0.190421
mortgage_rate_at_listing    0.164043
district                   -0.065484
ceiling_height              0.055678
building_type               0.038378
building_age               -0.032497
renovation                 -0.022489
has_balcony                -0.018111
floor                       0.016536
month_sin                  -0.014896
okrug                       0.013893
metro_line                  0.012619
month_cos                   0.008507
metro_station              -0.005742
living_share                0.005585
dow_cos                    -0.005279
seller_type                -0.002995
d

In [44]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

TARGET = 'price_rub'

# Убираем date-подобные
df_ml = df_sec_m.copy()
bad = [c for c in df_ml.columns 
       if pd.api.types.is_period_dtype(df_ml[c])
       or pd.api.types.is_datetime64_any_dtype(df_ml[c])]

if bad:
    print("Удаляем date-подобные:", bad)
    df_ml = df_ml.drop(columns=bad)
else:
    print("Date-подобных нет")
df_ml = df_ml.drop(columns=bad)

# Категориальные — оставляем как есть (строки/категории)
cat_cols = ['district', 'okrug', 'metro_station', 'metro_line',
            'building_type', 'renovation']
cat_cols = [c for c in cat_cols if c in df_ml.columns]

# ВАЖНО: для CatBoost категориальные должны быть строками или int, не category
for c in cat_cols:
    df_ml[c] = df_ml[c].astype(str)

# Циклические оставляем как числа — бустинг сам разберётся
X = df_ml.drop(columns=[TARGET])
y = df_ml[TARGET].values

# Split (тот же random_state, чтобы сравнение было честным)
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# log1p для целевой — как в нейросети
y_tr_log  = np.log1p(y_tr)
y_val_log = np.log1p(y_val)

print(f"X_tr: {X_tr.shape}, X_val: {X_val.shape}")
print(f"Категориальных: {len(cat_cols)}")

Date-подобных нет
X_tr: (40000, 27), X_val: (10000, 27)
Категориальных: 6


C:\Users\Computer\AppData\Local\Temp\ipykernel_21900\2697460062.py:10: DeprecationWarning: is_period_dtype is deprecated and will be removed in a future version. Use `isinstance(dtype, pd.PeriodDtype)` instead
  if pd.api.types.is_period_dtype(df_ml[c])


In [45]:
from catboost import CatBoostRegressor
import time

cat_idx = [X.columns.get_loc(c) for c in cat_cols]

model_cb = CatBoostRegressor(
    iterations=5000,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=3.0,
    loss_function='RMSE',
    eval_metric='MAPE',
    random_seed=42,
    early_stopping_rounds=200,
    verbose=200,
    cat_features=cat_idx,
)

t0 = time.time()
model_cb.fit(
    X_tr, y_tr_log,
    eval_set=(X_val, y_val_log),
    use_best_model=True,
)
print(f"Обучение: {time.time() - t0:.1f} сек")

0:	learn: 0.0293508	test: 0.0290607	best: 0.0290607 (0)	total: 150ms	remaining: 12m 27s
200:	learn: 0.0050139	test: 0.0049006	best: 0.0049006 (200)	total: 18.4s	remaining: 7m 18s
400:	learn: 0.0044698	test: 0.0044184	best: 0.0044184 (400)	total: 36.2s	remaining: 6m 55s
600:	learn: 0.0042437	test: 0.0042605	best: 0.0042605 (600)	total: 54.6s	remaining: 6m 39s
800:	learn: 0.0041067	test: 0.0041943	best: 0.0041943 (800)	total: 1m 11s	remaining: 6m 17s
1000:	learn: 0.0040039	test: 0.0041664	best: 0.0041664 (1000)	total: 1m 29s	remaining: 5m 56s
1200:	learn: 0.0039096	test: 0.0041476	best: 0.0041476 (1200)	total: 1m 45s	remaining: 5m 35s
1400:	learn: 0.0038239	test: 0.0041354	best: 0.0041354 (1399)	total: 2m 2s	remaining: 5m 15s
1600:	learn: 0.0037455	test: 0.0041264	best: 0.0041264 (1600)	total: 2m 19s	remaining: 4m 57s
1800:	learn: 0.0036760	test: 0.0041198	best: 0.0041197 (1799)	total: 2m 37s	remaining: 4m 39s
2000:	learn: 0.0036033	test: 0.0041178	best: 0.0041174 (1935)	total: 2m 54s	re

In [46]:
import lightgbm as lgb

# LightGBM требует категориальные как 'category'
X_tr_lgb = X_tr.copy()
X_val_lgb = X_val.copy()
for c in cat_cols:
    X_tr_lgb[c]  = X_tr_lgb[c].astype('category')
    X_val_lgb[c] = X_val_lgb[c].astype('category')

model_lgb = lgb.LGBMRegressor(
    n_estimators=5000,
    learning_rate=0.03,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=20,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1,
)

model_lgb.fit(
    X_tr_lgb, y_tr_log,
    eval_set=[(X_val_lgb, y_val_log)],
    eval_metric='mape',
    callbacks=[
        lgb.early_stopping(200, verbose=False),
        lgb.log_evaluation(200),
    ],
)

c:\Users\Computer\Desktop\programInj\house-price-predictor-\venv\lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001139 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2467
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 27
[LightGBM] [Info] Start training from score 16.203750
[200]	valid_0's mape: 0.00451148	valid_0's l2: 0.00846264
[400]	valid_0's mape: 0.00439244	valid_0's l2: 0.00796881
[600]	valid_0's mape: 0.00440032	valid_0's l2: 0.00798575


,boosting_type,'gbdt'
,num_leaves,63
,max_depth,-1
,learning_rate,0.03
,n_estimators,5000
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [47]:
import xgboost as xgb

X_tr_xgb = X_tr.copy()
X_val_xgb = X_val.copy()
for c in cat_cols:
    X_tr_xgb[c]  = X_tr_xgb[c].astype('category')
    X_val_xgb[c] = X_val_xgb[c].astype('category')

model_xgb = xgb.XGBRegressor(
    n_estimators=5000,
    learning_rate=0.03,
    max_depth=8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    tree_method='hist',
    enable_categorical=True,
    early_stopping_rounds=200,
)

model_xgb.fit(
    X_tr_xgb, y_tr_log,
    eval_set=[(X_val_xgb, y_val_log)],
    verbose=200,
)

[0]	validation_0-rmse:0.59494
[200]	validation_0-rmse:0.10357
[400]	validation_0-rmse:0.10084
[600]	validation_0-rmse:0.10071
[757]	validation_0-rmse:0.10079


,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,200
,enable_categorical,True
,eval_metric,None


In [48]:
def evaluate(name, model, X_val, y_val_true):
    pred_log = model.predict(X_val)
    pred = np.expm1(pred_log)
    true = np.expm1(y_val_true)

    mae  = np.mean(np.abs(pred - true))
    rmse = np.sqrt(np.mean((pred - true) ** 2))
    mape = np.mean(np.abs((pred - true) / true)) * 100
    r2   = 1 - np.sum((true - pred) ** 2) / np.sum((true - true.mean()) ** 2)

    print(f"\n=== {name} ===")
    print(f"MAE:  {mae:>12,.0f} руб")
    print(f"RMSE: {rmse:>12,.0f} руб")
    print(f"MAPE: {mape:>12.2f} %")
    print(f"R²:   {r2:>12.4f}")
    return {'mae': mae, 'rmse': rmse, 'mape': mape, 'r2': r2, 'pred': pred}


res_cb  = evaluate("CatBoost", model_cb,  X_val, y_val_log)
res_lgb = evaluate("LightGBM", model_lgb, X_val_lgb, y_val_log)
res_xgb = evaluate("XGBoost",  model_xgb, X_val_xgb, y_val_log)


=== CatBoost ===
MAE:       888,146 руб
RMSE:    1,349,683 руб
MAPE:         6.68 %
R²:         0.9753

=== LightGBM ===
MAE:       944,329 руб
RMSE:    1,442,982 руб
MAPE:         7.15 %
R²:         0.9717

=== XGBoost ===
MAE:     1,037,238 руб
RMSE:    1,652,607 руб
MAPE:         7.97 %
R²:         0.9629


In [49]:
baseline_nn = {'mape': 8.17, 'r2': 0.9655, 'mae': 1_036_322, 'rmse': 1_615_622}

results = pd.DataFrame({
    'NN (FlatNet)': baseline_nn,
    'CatBoost':     {k: v for k, v in res_cb.items()  if k != 'pred'},
    'LightGBM':     {k: v for k, v in res_lgb.items() if k != 'pred'},
    'XGBoost':      {k: v for k, v in res_xgb.items() if k != 'pred'},
}).T

print(results[['mape', 'r2', 'mae', 'rmse']].round(4))

                mape      r2           mae          rmse
NN (FlatNet)  8.1700  0.9655  1.036322e+06  1.615622e+06
CatBoost      6.6846  0.9753  8.881457e+05  1.349683e+06
LightGBM      7.1509  0.9717  9.443293e+05  1.442982e+06
XGBoost       7.9744  0.9629  1.037238e+06  1.652607e+06


In [50]:
pred_ensemble = (
    0.5 * res_cb['pred'] +
    0.3 * res_lgb['pred'] +
    0.2 * res_xgb['pred']
)

true = np.expm1(y_val_log)
mape_ens = np.mean(np.abs((pred_ensemble - true) / true)) * 100
print(f"Ensemble MAPE: {mape_ens:.2f}%")

Ensemble MAPE: 6.79%


---
← [Часть 2. Подготовка признаков и нормализация](02_features.ipynb) · [Все части](README.md) · [Часть 4. Эксперименты с CatBoost](04_catboost_experiments.ipynb) →